# 04 — Stacking Meta-Modelo (Ridge Regression)

Meta-modelo de stacking que combina las predicciones de los modelos base
entrenando una **Ridge Regression** directamente sobre `log_price`.

## Flujo

```
data/meta/test_*  +  train_processed.csv
        │
        ▼
  Ridge Regression  (entrena sobre log_price directamente)
        │
        ├──► submissions/test/  (Pippin_01.01 + 01.03 + 02.00 [+ 05.00])
        │          └──► submissions/test/Pippin_04.00.csv
        │
        └──► submissions/       (Pippin_01.01 + 01.03 + 02.00 [+ 05.00])
                   └──► submissions/Pippin_04.00.csv
```

## Mapeo de archivos

| Pippin file   | Feature equivalente | Transformación       | Condicional           |
|---------------|---------------------|----------------------|-----------------------|
| Pippin_01.01  | pred_sent_lgbm      | log(predicted_price) | siempre               |
| Pippin_01.03  | pred_lgbm           | log(predicted_price) | siempre               |
| Pippin_02.00  | pred_reg_log        | log(predicted_price) | siempre               |
| Pippin_05.00  | pred_xgboost        | log(predicted_price) | xgboost_included=True |

## 0. Imports, rutas y configuración

In [20]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Rutas base (relativas a la raíz del repo) ──────────────────────────────
BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
META_DIR  = os.path.join(BASE_DIR, 'data', 'meta') + os.sep
TAB_DIR   = os.path.join(BASE_DIR, 'data', 'tabular') + os.sep
STEST_DIR = os.path.join(BASE_DIR, 'submissions', 'test') + os.sep
SUB_DIR   = os.path.join(BASE_DIR, 'submissions') + os.sep

# ── Configuración del meta-modelo ──────────────────────────────────────────
xgboost_included = True   # True → agrega pred_xgboost como feature adicional

print('BASE_DIR        :', BASE_DIR)
print('xgboost_included:', xgboost_included)

BASE_DIR        : d:\GitHub\UA_MDM_Labo2_RS
xgboost_included: True


## 1. Cargar predicciones OOF — `data/meta/test_*`

Predicciones en escala logarítmica generadas por los modelos base sobre el split
de validación interno. Son las features del meta-modelo.
Si `xgboost_included=True` se añade también `test_xgboost.csv`.

In [21]:
test_sent = (
    pd.read_csv(META_DIR + 'test_sent_lgbm.csv')
    .rename(columns={'lgbm_pred': 'pred_sent_lgbm'})
)
test_lgbm = (
    pd.read_csv(META_DIR + 'test_lgbm.csv')
    .rename(columns={'lgbm_pred': 'pred_lgbm'})
)
test_reg = (
    pd.read_csv(META_DIR + 'test_regression.csv')
    [['zpid', 'regression_pred_log']]
    .rename(columns={'regression_pred_log': 'pred_reg_log'})
)

X_meta_df = test_sent.merge(test_lgbm, on='zpid').merge(test_reg, on='zpid')

if xgboost_included:
    test_xgb = pd.read_csv(META_DIR + 'test_xgboost.csv')
    pred_col = [c for c in test_xgb.columns if c != 'zpid'][0]
    test_xgb = test_xgb[['zpid', pred_col]].rename(columns={pred_col: 'pred_xgboost'})
    X_meta_df = X_meta_df.merge(test_xgb, on='zpid')
    print('XGBoost incluido en features de entrenamiento.')

FEATURE_COLS = [c for c in ['pred_sent_lgbm', 'pred_lgbm', 'pred_reg_log', 'pred_xgboost']
                if c in X_meta_df.columns]

print('Shape features meta (OOF):', X_meta_df.shape)
print('Features usadas:', FEATURE_COLS)
X_meta_df.head(3)

XGBoost incluido en features de entrenamiento.
Shape features meta (OOF): (2368, 5)
Features usadas: ['pred_sent_lgbm', 'pred_lgbm', 'pred_reg_log', 'pred_xgboost']


,zpid,pred_sent_lgbm,pred_lgbm,pred_reg_log,pred_xgboost
0,1013708,13.690957,13.644295,13.356154,13.661063
1,1005786,12.961164,12.917778,12.833630,12.915836
2,1013543,13.327622,13.309487,13.530240,13.335673


## 2. Obtener target real (`log_price`)

In [22]:
train_labels = pd.read_csv(TAB_DIR + 'train_processed.csv', usecols=['zpid', 'log_price'])

df_train = X_meta_df.merge(train_labels, on='zpid', how='inner')
print(f'Filas de entrenamiento del meta-modelo: {len(df_train)}')

X     = df_train[FEATURE_COLS].values
y_log = df_train['log_price'].values

print(f'Features: {FEATURE_COLS}')
print(f'log_price — min: {y_log.min():.3f}  max: {y_log.max():.3f}  mean: {y_log.mean():.3f}')

Filas de entrenamiento del meta-modelo: 2368
Features: ['pred_sent_lgbm', 'pred_lgbm', 'pred_reg_log', 'pred_xgboost']
log_price — min: 10.845  max: 14.479  mean: 13.032


## 3. Entrenar Ridge Regression

Entrena directamente sobre `log_price` (continuo), sin necesidad de binning.  
El parámetro `alpha` controla la regularización — valores más altos fuerzan los coeficientes
hacia cero, evitando que un solo modelo base domine el ensemble.

In [23]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

pipe.fit(X, y_log)

# Mostrar coeficientes (peso que el meta-modelo asigna a cada modelo base)
coefs = pipe.named_steps['ridge'].coef_
intercept = pipe.named_steps['ridge'].intercept_

print(f'Entrenamiento completado con {len(FEATURE_COLS)} features.')
print(f'Intercept: {intercept:.4f}')
print()
pd.DataFrame({'feature': FEATURE_COLS, 'coeficiente': coefs.round(4)})

Entrenamiento completado con 4 features.
Intercept: 13.0320



,feature,coeficiente
0,pred_sent_lgbm,0.1283
1,pred_lgbm,0.4028
2,pred_reg_log,0.0796
3,pred_xgboost,-0.0304


## 4. Evaluación en el conjunto de entrenamiento

In [24]:
log_pred_tr  = pipe.predict(X)
price_pred_tr = np.exp(log_pred_tr)
price_real_tr = np.exp(y_log)

print(f"MAE  log_price (train): {mean_absolute_error(y_log, log_pred_tr):.4f}")
print(f"RMSE log_price (train): {np.sqrt(mean_squared_error(y_log, log_pred_tr)):.4f}")
print(f"MAE  precio    (train): ${mean_absolute_error(price_real_tr, price_pred_tr):,.0f}")

MAE  log_price (train): 0.2265
RMSE log_price (train): 0.3315
MAE  precio    (train): $116,427


## 5. Helper: construir features desde archivos Pippin

Los archivos Pippin tienen `predicted_price` (escala real).  
Se convierte a log-escala: `log(predicted_price) ≈ lgbm_pred` (diff < 1e-5).  
Si `xgboost_included=True`, también carga `Pippin_05.00.csv`.

In [25]:
def build_features_from_pippin(folder):
    """
    Lee archivos Pippin de la carpeta indicada y construye la matriz de features
    para el meta-modelo (en log-escala).

    Mapeo fijo:
        Pippin_01.04.csv  =>  pred_sent_lgbm   (log)
        Pippin_01.03.csv  =>  pred_lgbm         (log)
        Pippin_02.00.csv  =>  pred_reg_log      (log)

    Condicional (xgboost_included=True):
        Pippin_05.00.csv  =>  pred_xgboost      (log)

    Returns: (zpid Series, X ndarray)
    """
    p01 = pd.read_csv(folder + 'Pippin_01.04.csv').rename(columns={'predicted_price': 'pred_sent_lgbm'})
    p03 = pd.read_csv(folder + 'Pippin_01.03.csv').rename(columns={'predicted_price': 'pred_lgbm'})
    p02 = pd.read_csv(folder + 'Pippin_02.00.csv').rename(columns={'predicted_price': 'pred_reg_log'})

    df = p01.merge(p03, on='zpid').merge(p02, on='zpid')

    if xgboost_included:
        p05 = pd.read_csv(folder + 'Pippin_05.00.csv').rename(columns={'predicted_price': 'pred_xgboost'})
        df = df.merge(p05, on='zpid')

    for col in FEATURE_COLS:
        df[col] = np.log(df[col])

    return df['zpid'], df[FEATURE_COLS].values

print('Helper definido.')
print('Features que usará:', FEATURE_COLS)

Helper definido.
Features que usará: ['pred_sent_lgbm', 'pred_lgbm', 'pred_reg_log', 'pred_xgboost']


## 6. Predicción A — `submissions/test/` → `Pippin_04.00.csv`

In [26]:
zpid_test, X_test = build_features_from_pippin(STEST_DIR)

out_a = pd.DataFrame({
    'zpid'            : zpid_test.values,
    'predicted_price' : np.round(np.exp(pipe.predict(X_test)), 2)
})

out_a.to_csv(STEST_DIR + 'Pippin_07.00.csv', index=False)
print(f'Guardado: {STEST_DIR}Pippin_07.00.csv  ({len(out_a)} filas)')
out_a.describe()

Guardado: d:\GitHub\UA_MDM_Labo2_RS\submissions\test\Pippin_07.00.csv  (11840 filas)


,zpid,predicted_price
count,1.184000e+04,1.184000e+04
mean,1.008799e+06,5.349534e+05
std,5.070509e+03,2.958875e+05
min,1.000001e+06,7.854191e+04
25%,1.004382e+06,3.176679e+05
50%,1.008754e+06,4.654691e+05
75%,1.013184e+06,6.950305e+05
max,1.017606e+06,1.757748e+06


## 7. Predicción B — `submissions/` → `Pippin_04.00.csv`

In [27]:
zpid_comp, X_comp = build_features_from_pippin(SUB_DIR)

out_b = pd.DataFrame({
    'zpid'            : zpid_comp.values,
    'predicted_price' : np.round(np.exp(pipe.predict(X_comp)), 2)
})

out_b.to_csv(SUB_DIR + 'Pippin_07.00.csv', index=False)
print(f'Guardado: {SUB_DIR}Pippin_07.00.csv  ({len(out_b)} filas)')
out_b.describe()

Guardado: d:\GitHub\UA_MDM_Labo2_RS\submissions\Pippin_07.00.csv  (5038 filas)


,zpid,predicted_price
count,5.038000e+03,5.038000e+03
mean,1.008814e+06,5.376474e+05
std,5.125186e+03,2.956000e+05
min,1.000004e+06,8.428182e+04
25%,1.004405e+06,3.170554e+05
50%,1.008930e+06,4.724593e+05
75%,1.013280e+06,6.965478e+05
max,1.017605e+06,1.685384e+06


## 8. Comparación con modelos base (`submissions/`)

In [28]:
p01 = pd.read_csv(SUB_DIR + 'Pippin_01.04.csv')
p03 = pd.read_csv(SUB_DIR + 'Pippin_01.03.csv')
p02 = pd.read_csv(SUB_DIR + 'Pippin_02.00.csv')

cmp = p01[['zpid']].copy()
cmp['sent_lgbm (01.04)'] = p01['predicted_price'].round(0).astype(int)
cmp['lgbm     (01.03)']  = p03['predicted_price'].round(0).astype(int)
cmp['regression (02)']   = p02['predicted_price'].round(0).astype(int)

if xgboost_included:
    p05 = pd.read_csv(SUB_DIR + 'Pippin_05.00.csv')
    cmp['xgboost (05.00)'] = p05['predicted_price'].round(0).astype(int)

cmp['META_Ridge (07.00)'] = out_b['predicted_price'].round(0).astype(int)

cmp.head(10)

,zpid,sent_lgbm (01.04),lgbm (01.03),regression (02),xgboost (05.00),META_Ridge (07.00)
0,1009018,465061,483372,408555,466667,469287
1,1001436,672479,690483,599009,712059,676551
2,1010988,510183,526217,571968,503699,530661
3,1002184,269351,280060,289037,285348,275847
4,1015717,319747,298384,272741,288421,297452
5,1001346,599202,590884,706591,579655,610336
6,1006728,137525,138055,152158,144772,136344
7,1002644,277204,251728,308848,255210,261357
8,1009198,791827,745873,839947,711867,776540
9,1008339,504837,512135,582426,488097,521484
